In [ ]:
import datetime as dt
import aiohttp
import pathlib
import pandas as pd
import matplotlib.pyplot as plt
import json
import energyid
import typing
from sklearn.linear_model import LinearRegression

from openenergyid.pvsim.elia import EliaPVSimulator
from openenergyid.elia import Region

# Set plt figure size
plt.rcParams["figure.figsize"] = (20, 6)

with open("secrets.json") as f:
    secrets = json.load(f)

In [ ]:
eid_client = energyid.PandasClient(
    client_id=secrets["EnergyID_CLIENTID"], client_secret=secrets["EnergyID_CLIENTSECRET"]
)
eid_client.authenticate(
    username=secrets["EnergyID_USERNAME"], password=secrets["EnergyID_PASSWORD"]
)

# PV Evolution

The aim of this notebook is to develop a method to assess the long term evolution of PV system performance.

To do this, we will use the Historical PV Load Factor, as published by Elia for each region in Belgium.
By performing a linear regression on the first year of data from a system (the "training period", or "reference period"), we can establish a so-called "actual size" for the system. Which is to say: if the load factor is X, the output of the system is Y. This number can be expressed in kWp-equivalent.
By then determining the kWp-equivalent for each subsequent period, we can determine whether the system is performing better or worse than expected, and by how much.

## Historical PV Load Factor

Let's start by loading the historical PV load factor data.

Note: this data is only available since 2020-07-01.

In [ ]:
start = dt.date(2020, 7, 1)
end = dt.date(2026, 1, 1)

region = Region.East_Flanders

In [ ]:
filename = pathlib.Path(
    f"data/elia_load_factors_{start.isoformat()}_{end.isoformat()}_{region.value}.parquet.gz"
)

if filename.exists():
    elia_load_factors = pd.read_parquet(filename)
else:
    async with aiohttp.ClientSession() as session:
        elia_load_factors = await EliaPVSimulator.download_load_factors(
            start=start, end=end, region=region, session=session
        )
        elia_load_factors = elia_load_factors.to_frame()
        elia_load_factors.to_parquet(filename, compression="gzip")

In [ ]:
elia_load_factors["loadfactor"] = elia_load_factors["loadfactor"] / 100

In [ ]:
elia_load_factors.plot(alpha=0.5)

In [ ]:
# Split into calendar years and plot each year as a separate line
# But, reset the index to be the day of the year, so that the lines are comparable
tmp = elia_load_factors.copy()

tmp = tmp.resample("W-MON").mean()

new_index = tmp.index.dayofyear
tmp["year"] = tmp.index.year
tmp = tmp.set_index(new_index)
fig = tmp.groupby("year")["loadfactor"].plot(legend=True, alpha=0.5)

# Sample PV system

Let's download the data for a sample PV system, and plot it.

In [ ]:
record = eid_client.get_record(record_id=secrets["RECORDS"]["PVSYSTEM"])

# This specific record has data from 2021-11-01
start = dt.date(2021, 11, 1)

# The rated power of the system is 147 kWp

In [ ]:
filename = pathlib.Path(
    f"data/pv_production_{start.isoformat()}_{end.isoformat()}_PVSYSTEM.parquet.gz"
)

if filename.exists():
    pv_production = pd.read_parquet(filename)
else:
    pv_production = record.get_data(
        name="energyProduction", start=start.isoformat(), end=end.isoformat(), interval="PT15M"
    )
    pv_production = typing.cast(pd.Series, pv_production)
    pv_production = pv_production.to_frame()
    pv_production.to_parquet(filename, compression="gzip")

In [ ]:
pv_production.resample("MS").sum().plot.bar()

In [ ]:
pv_production["power"] = pv_production["energyProduction"] * 4

## Merge

In [ ]:
df = pd.merge(pv_production, elia_load_factors, left_index=True, right_index=True, how="left")

In [ ]:
df.dropna(how="any", inplace=True)

In [ ]:
# Training dataset, which is the first year

train = df.truncate(
    before=pd.Timestamp("2022-01-01", tz="Europe/Brussels"),
    after=pd.Timestamp("2023-01-01", tz="Europe/Brussels") - pd.Timedelta("1s"),
)

In [ ]:
train

# Getting to kWp-equivalent

## 1. Simple Linear regression

### PT15M

In [ ]:
def do_linreg(df: pd.DataFrame, x_name: str = "loadfactor", y_name: str = "power"):
    reg = LinearRegression(fit_intercept=False, positive=True).fit(X=df[[x_name]], y=df[y_name])
    coef = reg.coef_[0]
    score = reg.score(X=df[[x_name]], y=df[y_name])
    print(f"Coefficient: {coef:.4f}")
    print(f"R^2 score: {score:.4f}")

    # Scatter plot of load factor vs production
    df.plot.scatter(x=x_name, y=y_name)

    # Plot the regression line
    x = df[x_name]
    y = reg.predict(df[[x_name]])
    plt.plot(x, y, color="red")
    plt.show()

In [ ]:
do_linreg(train)

### PT1H

In [ ]:
tmp = train.resample("h").mean()
do_linreg(tmp)

### P1D

In [ ]:
tmp = train.resample("1D").mean()
do_linreg(tmp)

### P1W-MON

In [ ]:
tmp = train.resample("W-MON").mean()
do_linreg(tmp)

### P1M

In [ ]:
tmp = train.resample("MS").mean()
do_linreg(tmp)

# Binning the load factors

In [ ]:
def do_binned_linreg(df: pd.DataFrame, remove_outliers: bool = False):
    # Let's put the load factor in  bins and compute the median power for each bin
    tmp = df.copy()

    tmp["loadfactor_bin"] = pd.cut(tmp["loadfactor"], bins=100)

    train_binned = tmp.groupby("loadfactor_bin", observed=True)["power"].median().reset_index()
    train_binned["midpoint"] = train_binned["loadfactor_bin"].apply(lambda x: x.mid)

    if remove_outliers:
        for _ in range(100):
            train_binned = train_binned[train_binned["power"].diff().fillna(0) >= 0]

    do_linreg(train_binned, x_name="midpoint", y_name="power")

In [ ]:
do_binned_linreg(train)

In [ ]:
# Get rid of outliers by dropping rows if the power is lower or equal to the previous row, which is not expected in a cumulative production curve
do_binned_linreg(train, remove_outliers=True)

# Other Years

In [ ]:
inference = df.truncate(before=pd.Timestamp("2023-01-01", tz="Europe/Brussels"))

In [ ]:
for year, group in inference.groupby(inference.index.year):
    print(f"Year: {year}")
    do_binned_linreg(group, remove_outliers=True)